# Réduire les accidents de la route — pipeline d'aide à la décision pour une agence de voirie

**Rôle incarné :** agence de voirie / DOT d'une collectivité territoriale (gestionnaire d'infrastructure routière).
**Données :** *US Accidents* (Feb 2016 → Mar 2023, ~7,73 M accidents, 49 États).

---

## Le problème métier

> *« Comment réduire durablement les accidents sur notre réseau, avec un budget de travaux limité ? »*

Un DOT ne peut pas tout rénover. Il doit décider **où** intervenir et **quoi** y faire. Aujourd'hui cette
priorisation est surtout réactive (remontées terrain, accidents médiatisés). On apporte une priorisation
**data-driven et explicable**. On décline le fil rouge en trois questions :

| Question métier | Traduction data | Nature |
|---|---|---|
| **OÙ** rénover en priorité ? | Classement des zones par risque, **normalisé par l'exposition** routière | Agrégation géospatiale (descriptif) |
| **QUOI** installer sur place ? | Quels éléments d'infra aggravent → **modèle de gravité interprété** | ML supervisé *(moteur de preuve)* |
| **QUAND** activer des mesures temporaires ? | Patterns temporels / météo du risque | Visualisation descriptive |

## Le principe méthodologique clé : le modèle comme *moteur de preuve*

On **n'a pas** pour objectif de livrer un prédicteur de gravité. Notre but est de **corriger** le problème.
Mais pour savoir *quoi* corriger, on entraîne un modèle qui prédit la gravité d'un accident à partir de
l'infrastructure et du contexte, **puis on l'interroge** (importances de variables, valeurs SHAP) pour
identifier *quels éléments structurels sont associés à des accidents plus graves*.

> **La prédiction est le moyen ; l'explication est le livrable.** Ce détour se justifie : il fournit une
> tâche supervisée *évaluable* (on peut comparer plusieurs modèles), il *contrôle les confondants*
> (météo/heure mis comme variables de contrôle), et il évite la **tautologie** d'un modèle qui prédirait une
> quantité à partir de ses propres composantes.

**Ce qu'on ne prétend pas :** prouver une causalité. Sans panel avant/après installation d'équipement, nos
résultats sont des **associations / signaux de priorisation**, pas des preuves d'efficacité. (cf. § *Limites*.)

## Architecture de la pipeline

```
Setup → Chargement (schéma explicite) → NETTOYAGE explicite
   ├─ A. EDA / Visualisation (volumétrie, biais, sévérité, "QUAND")
   ├─ B. OÙ  : scoring H3 + exposition OSM → zones prioritaires
   ├─ C. QUOI: modèle de gravité (baseline→logreg→arbre→RF→GBT) → SHAP → leviers
   ├─ D. Synthèse décisionnelle (par zone : où + quoi + impact estimé)
   └─ E. Analyse critique & recul
```

Le notebook s'exécute **de bout en bout sans intervention** (*Run All*). Les données sont téléchargées
automatiquement, les graines aléatoires sont fixées, et le nettoyage est **défensif** : il fonctionne sur
n'importe quel sous-ensemble du dataset (n'importe quel État, n'importe quelle période).

## 0 — Setup & reproductibilité

On fixe l'environnement pour que **n'importe qui obtienne le même résultat**. Toutes les dépendances sont
dans `requirements.txt` (versions figées). Les constantes de la pipeline sont **rassemblées ici** et nommées :
changer `STATE` suffit à rejouer toute l'analyse sur un autre territoire → c'est notre preuve de **généricité**.

In [ ]:
# Installe les dépendances (idempotent). Décommenter si l'environnement n'est pas déjà prêt.
# %pip install -q -r requirements.txt

In [ ]:
import os, json, warnings, math, random
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- Graines : reproductibilité ---
SEED = 42
random.seed(SEED); np.random.seed(SEED)

# --- Constantes de pipeline (tout est paramétrable et documenté ici) ---
STATE          = "CA"     # territoire de démonstration ; changer suffit à rejouer ailleurs
H3_RES         = 8        # ~0,7 km² : l'échelle d'un carrefour
PRIORITY_N     = 50       # nb de zones prioritaires retenues
LIFT_MIN       = 1.5      # seuil de sur-représentation pour attribuer un aléa à une zone
PRESENCE_MIN   = 0.05     # un équipement est "présent" s'il touche >= 5% des accidents de la zone
MIN_N          = 200      # taille mini d'échantillon pour un effet conditionnel fiable
GRAVE_THRESHOLD = 3       # sévérité >= 3 => accident "grave" (cible binaire)
MAX_MODEL_ROWS = 800_000  # plafond de lignes pour la comparaison de modèles (compromis repro/coût ; None = tout)
RUN_OSM_EXPOSURE = True    # exposition routière OSM (réseau requis) ; bascule à False si hors-ligne
SHAP_SAMPLE    = 5_000    # nb de lignes pour le calcul SHAP (TreeExplainer ne scale pas à des millions)

print("Config :", dict(STATE=STATE, H3_RES=H3_RES, PRIORITY_N=PRIORITY_N,
                        GRAVE_THRESHOLD=GRAVE_THRESHOLD, MAX_MODEL_ROWS=MAX_MODEL_ROWS))

**Pourquoi PySpark ?** Le dataset fait 7,73 M lignes : pandas pur saturerait la mémoire pour les étapes
globales (chargement, agrégation US). Spark traite ce volume en local (`local[*]`) et scalerait vers un
cluster sans changer le code. On bascule en pandas/scikit-learn **seulement** quand le volume est réduit
(agrégats par zone, échantillon de modélisation).

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (StructType, StructField, StringType, IntegerType,
                               DoubleType, BooleanType, TimestampType)

spark = (SparkSession.builder
         .master("local[*]").appName("voirie_reduction_accidents")
         .config("spark.driver.memory", "6g")
         .config("spark.sql.session.timeZone", "UTC")
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")
print("Spark", spark.version)

On gère la compatibilité **h3 v3/v4** une seule fois (factorisation : ces helpers sont réutilisés
partout dans le notebook, jamais recopiés).

In [ ]:
import h3
try:
    h3.latlng_to_cell(0.0, 0.0, H3_RES)            # API v4
    h3_encode   = lambda lat, lng: h3.latlng_to_cell(lat, lng, H3_RES)
    h3_boundary = lambda c: h3.cell_to_boundary(c)
    h3_center   = lambda c: h3.cell_to_latlng(c)
except AttributeError:                              # API v3
    h3_encode   = lambda lat, lng: h3.geo_to_h3(lat, lng, H3_RES)
    h3_boundary = lambda c: h3.h3_to_geo_boundary(c)
    h3_center   = lambda c: h3.h3_to_geo(c)
print("h3", h3.__version__)

## 1 — Chargement (reproductible, schéma explicite)

Données chargées **automatiquement** via `kagglehub` (repli sur un CSV local s'il existe). On déclare un
**schéma explicite** plutôt que `inferSchema` : (1) reproductibilité — les types ne dépendent pas d'un
échantillon lu au hasard ; (2) performance — on évite une passe de lecture supplémentaire.

In [ ]:
LOCAL_CANDIDATES = [
    "US_Accidents_March23.csv", "../US_Accidents_March23.csv",
    os.path.expanduser("~/Prog/Epita/ING2/xPloring/US_Accidents_March23.csv"),
]
CSV_PATH = next((p for p in LOCAL_CANDIDATES if os.path.exists(p)), None)
if CSV_PATH is None:
    import kagglehub
    path = kagglehub.dataset_download("sobhanmoosavi/us-accidents")
    csvs = [f for f in os.listdir(path) if f.endswith(".csv")]
    assert csvs, "Aucun CSV dans le répertoire KaggleHub."
    CSV_PATH = os.path.join(path, csvs[0])
print("CSV utilisé :", CSV_PATH)

In [ ]:
schema = StructType([
    StructField("ID", StringType()), StructField("Source", StringType()),
    StructField("Severity", IntegerType()),
    StructField("Start_Time", TimestampType()), StructField("End_Time", TimestampType()),
    StructField("Start_Lat", DoubleType()), StructField("Start_Lng", DoubleType()),
    StructField("End_Lat", DoubleType()), StructField("End_Lng", DoubleType()),
    StructField("Distance(mi)", DoubleType()), StructField("Description", StringType()),
    StructField("Street", StringType()), StructField("City", StringType()),
    StructField("County", StringType()), StructField("State", StringType()),
    StructField("Zipcode", StringType()), StructField("Country", StringType()),
    StructField("Timezone", StringType()), StructField("Airport_Code", StringType()),
    StructField("Weather_Timestamp", TimestampType()),
    StructField("Temperature(F)", DoubleType()), StructField("Wind_Chill(F)", DoubleType()),
    StructField("Humidity(%)", DoubleType()), StructField("Pressure(in)", DoubleType()),
    StructField("Visibility(mi)", DoubleType()), StructField("Wind_Direction", StringType()),
    StructField("Wind_Speed(mph)", DoubleType()), StructField("Precipitation(in)", DoubleType()),
    StructField("Weather_Condition", StringType()),
    StructField("Amenity", BooleanType()), StructField("Bump", BooleanType()),
    StructField("Crossing", BooleanType()), StructField("Give_Way", BooleanType()),
    StructField("Junction", BooleanType()), StructField("No_Exit", BooleanType()),
    StructField("Railway", BooleanType()), StructField("Roundabout", BooleanType()),
    StructField("Station", BooleanType()), StructField("Stop", BooleanType()),
    StructField("Traffic_Calming", BooleanType()), StructField("Traffic_Signal", BooleanType()),
    StructField("Turning_Loop", BooleanType()), StructField("Sunrise_Sunset", StringType()),
    StructField("Civil_Twilight", StringType()), StructField("Nautical_Twilight", StringType()),
    StructField("Astronomical_Twilight", StringType()),
])
raw = spark.read.option("header", True).schema(schema).csv(CSV_PATH)
# snake_case : '(', ')', '%' -> séparateurs propres
raw = raw.toDF(*[c.lower().replace("(", "_").replace(")", "").replace("%", "pct") for c in raw.columns])
print("Lignes :", f"{raw.count():,}", "| colonnes :", len(raw.columns))

## 2 — Nettoyage & prétraitement (explicite, défensif, justifié)

> Exigence du projet : **rien de caché**. Pour chaque décision : *quoi* + *pourquoi*. Le nettoyage est
> écrit comme **une fonction unique** appliquée à n'importe quel sous-ensemble (n'importe quel État/période),
> ce qui garantit que la pipeline « rend viables » des données brutes quelconques du dataset.

**Décisions de nettoyage et leurs raisons :**

| Élément | Décision | Pourquoi |
|---|---|---|
| `end_lat`, `end_lng` (~44 % NA) | **abandonnés** | Trop manquants ; on travaille sur le point de départ `start_lat/lng`. |
| `wind_chill_f` (~26 % NA) | **supprimé** | Redondant avec `temperature_f`. |
| Météo numérique (NA résiduels) | **imputation médiane** | Robuste aux valeurs extrêmes ; conserve l'échantillon. |
| `weather_condition` (texte libre, ~100 modalités) | **regroupé** en 6 buckets | Évite l'explosion de variables ; modalités lisibles (Clear/Cloud/Rain/Snow/Fog/Other). |
| Lignes sans `severity`/`start_lat`/`start_lng` | **supprimées** | Inutilisables (cible ou localisation absente). |
| Booléens d'infra NA | **`False`** | NA = équipement non signalé ⇒ traité comme absent. |
| `severity ∈ {1..4}` | **binarisé** `grave = severity ≥ 3` | Classe 2 ≈ 80 % : le multiclasse rendrait métriques et message illisibles. |
| `turning_loop` | **exclu** des features | Quasi constant ⇒ aucun pouvoir discriminant. |

On dérive aussi des **variables temporelles** (`hour`, `dow`, `month`, `is_night`) : elles servent au volet
« QUAND » (visu) et de **variables de contrôle** dans le modèle.

In [ ]:
INFRA = ["junction", "crossing", "stop", "traffic_signal", "railway", "station",
         "roundabout", "bump", "give_way", "no_exit", "traffic_calming", "amenity"]
WEATHER_NUM = ["temperature_f", "humidity_pct", "visibility_mi", "wind_speed_mph", "pressure_in"]

def bucket_weather(col):
    # Regroupe le texte libre météo en 6 familles lisibles (défensif : NULL -> 'Other').
    c = F.lower(F.coalesce(col, F.lit("")))
    return (F.when(c.contains("snow") | c.contains("sleet") | c.contains("wintry") | c.contains("ice"), "Snow")
             .when(c.contains("rain") | c.contains("drizzle") | c.contains("shower") | c.contains("thunder"), "Rain")
             .when(c.contains("fog") | c.contains("mist") | c.contains("haze"), "Fog")
             .when(c.contains("cloud") | c.contains("overcast"), "Cloud")
             .when(c.contains("clear") | c.contains("fair"), "Clear")
             .otherwise("Other"))

def clean(df):
    # Nettoyage défensif appliqué à n'importe quel sous-ensemble du dataset.
    df = df.drop("end_lat", "end_lng", "wind_chill_f")            # trop de NA / redondant
    df = df.dropna(subset=["severity", "start_lat", "start_lng"]) # cible & localisation requises
    # cible binaire
    df = df.withColumn("grave", (F.col("severity") >= F.lit(GRAVE_THRESHOLD)).cast("int"))
    # dérivées temporelles
    df = (df.withColumn("hour", F.hour("start_time"))
            .withColumn("dow", F.dayofweek("start_time"))
            .withColumn("month", F.month("start_time"))
            .withColumn("year", F.year("start_time"))
            .withColumn("is_night", (F.col("sunrise_sunset") == F.lit("Night")).cast("int")))
    # météo regroupée + booléens infra -> 0/1 (NA = absent)
    df = df.withColumn("weather_bucket", bucket_weather(F.col("weather_condition")))
    for c in INFRA:
        df = df.withColumn(c, F.coalesce(F.col(c), F.lit(False)).cast("int"))
    for c in ["hour", "is_night"]:
        df = df.withColumn(c, F.coalesce(F.col(c), F.lit(0)))
    return df

clean_df = clean(raw).cache()
n_clean = clean_df.count()
print(f"Après nettoyage : {n_clean:,} lignes")
print("Taux d'accidents graves (cible) :",
      f"{clean_df.agg(F.mean('grave')).first()[0]*100:.1f} %  -> classe déséquilibrée, à gérer (§C)")

## A — Exploration & visualisation (dont « QUAND »)

Chaque visualisation répond à une **question métier** précise (pas de décor). On met aussi en avant les
résultats **contre-intuitifs** : ce sont eux qui créent de la valeur par rapport au *common knowledge*.

### A.1 Couverture & biais — *« peut-on faire confiance à ces données partout ? »*

In [ ]:
by_state = (clean_df.groupBy("state").count().orderBy(F.desc("count"))
            .limit(15).toPandas())
tot = clean_df.count()
top3 = by_state["count"].head(3).sum() / tot * 100
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(by_state["state"], by_state["count"])
ax.set_title(f"Top 15 États par volume — Top 3 = {top3:.0f}% du dataset (couverture très déséquilibrée)")
ax.set_ylabel("accidents"); plt.tight_layout(); plt.show()

**Lecture :** la couverture est très inégale (Top 3 ≈ 41 %). Conséquence assumée (*biais de reporting*
MapQuest/Bing) : *peu de lignes ≠ peu d'accidents*. On ne compare donc pas les États entre eux ; on raisonne
**à l'intérieur d'un territoire** (d'où l'analyse « OÙ » paramétrée par `STATE`).

### A.2 Distribution de la sévérité — *« justifie la binarisation »*

In [ ]:
sev = clean_df.groupBy("severity").count().orderBy("severity").toPandas()
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(sev["severity"].astype(str), sev["count"])
ax.set_title("Distribution de la sévérité (1..4) — la classe 2 écrase tout")
ax.set_xlabel("severity"); ax.set_ylabel("accidents"); plt.tight_layout(); plt.show()
print((sev.assign(pct=lambda d: (d["count"]/d["count"].sum()*100).round(1))))

**Lecture :** la sévérité 2 domine (~80 %). Une classification 1–4 produirait des métriques illisibles
et un message flou. On **binarise** (`grave = sévérité ≥ 3`), ce qui donne une cible nette et une métrique
interprétable côté décideur (« grave / non grave »).

### A.3 « QUAND » — patterns temporels & météo (pour les mesures dynamiques)

In [ ]:
hourly = clean_df.groupBy("hour").agg(F.count("*").alias("n"),
                                       F.mean("grave").alias("part_grave")).orderBy("hour").toPandas()
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].bar(hourly["hour"], hourly["n"]);          ax[0].set_title("Volume par heure"); ax[0].set_xlabel("heure")
ax[1].plot(hourly["hour"], hourly["part_grave"]); ax[1].set_title("Part d'accidents graves par heure")
ax[1].set_xlabel("heure"); ax[1].set_ylabel("part grave"); plt.tight_layout(); plt.show()

In [ ]:
wx = (clean_df.groupBy("weather_bucket")
      .agg(F.count("*").alias("n"), F.mean("grave").alias("part_grave"))
      .orderBy(F.desc("n")).toPandas())
print("Sévérité par condition météo (regroupée) :")
print(wx.assign(part_grave=lambda d: (d["part_grave"]*100).round(1)))

**Lecture (« QUAND ») :** le risque varie par heure (pics aux heures de pointe). Côté météo, le message
simpliste « la pluie tue » est souvent **contredit** : la part d'accidents graves n'explose pas par mauvais
temps (hypothèse : on roule plus vite par temps clair → chocs plus violents). Ces signaux relèvent du **risque
conjoncturel** → ils nourrissent d'éventuelles **mesures dynamiques** (panneaux à messages variables), et non
la rénovation structurelle. On ne les modélise donc pas dans le cœur ML, on les **visualise**.

## B — OÙ rénover en priorité (scoring géospatial + exposition)

L'unité d'analyse n'est plus l'accident mais la **zone** (cellule H3, rés. 8, **équi-surface** → un comptage
est déjà une densité comparable). On classe les zones d'un territoire (`STATE`) par leur **charge
d'accidents pondérée par la gravité**, puis on **normalise par l'exposition routière** pour distinguer
« vraiment dangereux » de « simplement très fréquenté ».

- **Pourquoi pondérer par la gravité ?** 100 accidents légers ≠ 100 accidents graves.
- **Pourquoi filtrer par le volume et non la gravité seule ?** un fort volume au même endroit = problème
  *structurel* (géométrie, équipement) ; un accident grave isolé = *erreur humaine ponctuelle*, hors levier DOT.

In [ ]:
# On ramène le territoire choisi en pandas (volume réduit : ~1,7 M pour la CA)
terr = (clean_df.filter(F.col("state") == STATE)
        .select("start_lat", "start_lng", "severity", "grave", *INFRA)).toPandas()
terr["h3_cell"] = [h3_encode(a, b) for a, b in zip(terr["start_lat"], terr["start_lng"])]
print(f"{STATE} : {len(terr):,} accidents, {terr['h3_cell'].nunique():,} cellules H3")

In [ ]:
# Agrégation par zone, pondérée gravité (charge = somme des sévérités).
for el in INFRA:
    terr["sev_" + el] = terr["severity"] * terr[el]
aggmap = {"severity": ["count", "mean", "sum"]}
for el in INFRA:
    aggmap[el] = "sum"; aggmap["sev_" + el] = "sum"
zone = terr.groupby("h3_cell").agg(aggmap)
zone.columns = ["n_accidents", "avg_severity", "charge"] + \
               [c for el in INFRA for c in ("n_" + el, "charge_" + el)]
zone = zone.reset_index().sort_values("charge", ascending=False).reset_index(drop=True)
zone["rank"] = zone.index + 1
priority = zone.head(PRIORITY_N).copy()
print(f"{len(zone):,} zones agrégées — top {PRIORITY_N} retenu comme prioritaire (par charge)")
priority[["rank", "n_accidents", "avg_severity", "charge"]].head(8)

### B.1 Exposition routière (OSM) — *« dangereux, ou juste fréquenté ? »*

On télécharge, pour chaque zone prioritaire, la **longueur de réseau routier** (OSM via `osmnx`) contenue
dans son hexagone, et on calcule `accidents/km` et `charge/km`. C'est le **risque réel à offre routière
donnée**. Étape encapsulée dans un `try/except` avec drapeau `RUN_OSM_EXPOSURE` : le notebook reste
exécutable **même hors-ligne** (on retombe alors proprement sur le classement par volume).

In [ ]:
def osm_street_km(cell):
    # Longueur de voirie (km) dans l'hexagone H3, via OSM. None si indisponible (ex : osmnx absent / hors-ligne).
    try:
        import osmnx as ox
        from shapely.geometry import Polygon
        b = h3_boundary(cell)                   # [(lat, lng), ...]
        poly = Polygon([(lng, lat) for lat, lng in b])
        G = ox.graph_from_polygon(poly, network_type="drive", retain_all=True)
        return sum(d.get("length", 0.0) for *_ , d in G.edges(data=True)) / 1000.0
    except Exception:
        return None

priority["street_km"] = np.nan
if RUN_OSM_EXPOSURE:
    cache = {}
    for i, c in enumerate(priority["h3_cell"]):
        if c not in cache:
            cache[c] = osm_street_km(c)
        priority.iloc[i, priority.columns.get_loc("street_km")] = cache[c]
    ok = priority["street_km"].notna().sum()
    print(f"Exposition OSM récupérée pour {ok}/{len(priority)} zones")
else:
    print("Exposition OSM désactivée (RUN_OSM_EXPOSURE=False) -- classement par volume conservé.")

# Taux par exposition (NaN si OSM indisponible -> géré en aval)
priority["acc_per_km"]    = priority["n_accidents"] / priority["street_km"]
priority["charge_per_km"] = priority["charge"]      / priority["street_km"]

In [ ]:
# Effet de la normalisation : qui monte / descend au classement ?
if priority["street_km"].notna().any():
    tmp = priority[priority["street_km"].notna()].copy()
    tmp["rank_charge"]  = tmp["charge"].rank(ascending=False)
    tmp["rank_per_km"]  = tmp["charge_per_km"].rank(ascending=False)
    tmp["mouvement"]    = tmp["rank_charge"] - tmp["rank_per_km"]   # >0 : monte après normalisation
    print("Plus gros mouvements après normalisation par l'exposition :")
    cols = ["rank", "n_accidents", "charge", "street_km", "charge_per_km", "mouvement"]
    print(tmp.sort_values("mouvement", ascending=False)[cols].head(6).round(2))
else:
    print("Pas d'exposition disponible : on conserve le classement par charge.")

**Lecture :** certaines zones très accidentées le sont d'abord parce qu'elles concentrent beaucoup de
voirie (forte exposition) ; normalisées par km, elles redescendent. À l'inverse, des zones plus modestes en
volume mais à **forte densité d'accidents par km** remontent : ce sont des **points noirs concentrés**,
prioritaires pour la rénovation. (Si OSM est indisponible, on reste sur la charge brute, documenté comme limite.)

## C — QUOI installer : le modèle de gravité comme *moteur de preuve*

On entraîne, **sur tout le US** (généricité), un modèle qui prédit `grave` (0/1) à partir de :

- **leviers d'action** — les 12 booléens d'infrastructure (ce qu'on *interprétera* pour décider quoi installer) ;
- **variables de contrôle** — météo, heure, jour, mois, nuit, État (pour *isoler* l'effet propre de l'infra
  des confondants ; on ne les interprète **pas** comme des leviers : on ne rénove pas la pluie).

> **Anti-fuite :** aucune feature n'est dérivée de la cible. La gravité d'un accident est prédite à partir du
> *contexte* de cet accident, pas d'une quantité qui la contient.

In [ ]:
ACTION_FEATURES  = INFRA                                  # interprétés -> "quoi installer"
NUM_CONTROLS     = WEATHER_NUM + ["hour", "dow", "month"]  # contrôles numériques
CAT_CONTROLS     = ["weather_bucket", "state", "is_night"] # contrôles catégoriels
ALL_FEATURES     = ACTION_FEATURES + NUM_CONTROLS + CAT_CONTROLS
TARGET           = "grave"

# Table de modélisation US-wide, avec l'année pour le split temporel
model_cols = ALL_FEATURES + [TARGET, "year"]
mdf = clean_df.select(*model_cols)

# Split TEMPOREL (train < 2022, test 2022-2023) : simule l'usage réel (prédire le futur avec le passé)
train_sdf = mdf.filter(F.col("year") < 2022)
test_sdf  = mdf.filter(F.col("year") >= 2022)

# Échantillonnage maîtrisé pour la comparaison sklearn (compromis repro/coût, explicité).
def to_pandas_capped(sdf, cap):
    n = sdf.count()
    frac = 1.0 if (cap is None or n <= cap) else cap / n
    out = (sdf if frac >= 1.0 else sdf.sample(False, frac, seed=SEED)).toPandas()
    return out, n, frac

train_pd, n_tr, f_tr = to_pandas_capped(train_sdf, MAX_MODEL_ROWS)
test_pd,  n_te, f_te = to_pandas_capped(test_sdf,  MAX_MODEL_ROWS // 4)
print(f"Train US : {n_tr:,} lignes (échantillon {len(train_pd):,}, frac={f_tr:.3f})")
print(f"Test  US : {n_te:,} lignes (échantillon {len(test_pd):,}, frac={f_te:.3f})")
print("Part grave — train:", round(train_pd[TARGET].mean(), 3),
      "| test:", round(test_pd[TARGET].mean(), 3))

**Pourquoi un échantillon (et pas 7,7 M lignes brutes dans scikit-learn) ?** Pour que le notebook reste
**exécutable de bout en bout** en un temps raisonnable tout en restant représentatif. L'échantillonnage est
*stratifié dans le temps* (le split temporel est fait **avant**), reproductible (`seed`), et plafonné par la
constante `MAX_MODEL_ROWS` (mettre `None` pour utiliser tout le dataset). Le code est **agnostique au
territoire** : la généricité tient au code, pas à un sous-ensemble figé.

### C.1 Prétraitement encodé dans un `Pipeline` (pas de fuite test→train)

On encode dans un `ColumnTransformer` : imputation médiane + standardisation des numériques (utile à la
régression logistique, neutre pour les arbres), one-hot des catégorielles (`handle_unknown='ignore'` →
robuste à une modalité jamais vue), passthrough des booléens d'infra (déjà 0/1, et noms préservés pour SHAP).

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (average_precision_score, roc_auc_score, f1_score,
                             recall_score, confusion_matrix)
from sklearn.utils.class_weight import compute_sample_weight

num_pipe = Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())])
cat_pipe = Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                     ("oh", OneHotEncoder(handle_unknown="ignore", min_frequency=20))])
prep = ColumnTransformer([
    ("num",   num_pipe, NUM_CONTROLS),
    ("cat",   cat_pipe, CAT_CONTROLS),
    ("infra", "passthrough", ACTION_FEATURES),
])

X_tr, y_tr = train_pd[ALL_FEATURES], train_pd[TARGET]
X_te, y_te = test_pd[ALL_FEATURES],  test_pd[TARGET]
# Poids pour gérer le déséquilibre SANS fabriquer de données (préféré à SMOTE).
sw_tr = compute_sample_weight("balanced", y_tr)

### C.2 Comparaison de modèles — *tester plusieurs, justifier le choix*

On compare cinq modèles, **dans des conditions identiques** (même prétraitement, même split, même métrique).
Chacun a un rôle dans l'argumentaire :

| Modèle | Rôle | Pourquoi |
|---|---|---|
| `DummyClassifier` | **plancher** | situe toutes les métriques (« ne rien apprendre »). |
| `LogisticRegression` | baseline **linéaire interprétable** | si les arbres ne la battent pas, la complexité n'est pas justifiée. |
| `DecisionTree` | **whitebox** | entièrement transparent (règles lisibles). |
| `RandomForest` | ensemble baggé | réduit la variance, importances + SHAP natif. |
| `HistGradientBoosting` | ensemble boosté | souvent le meilleur sur tabulaire, gère le volume. |

**Métrique (attention au piège) :** on **n'utilise pas l'accuracy** (à 80/20, prédire « non grave » partout
donne 80 % sans rien apprendre). On regarde **PR-AUC** (priorité : classe positive rare), **ROC-AUC**, et le
**rappel sur la classe grave** (un grave manqué = un point noir non détecté). On compare tout au plancher.

In [ ]:
MODELS = {
    "Dummy (plancher)":      DummyClassifier(strategy="most_frequent"),
    "LogReg (linéaire)":     LogisticRegression(max_iter=2000),
    "DecisionTree (whitebox)": DecisionTreeClassifier(max_depth=6, random_state=SEED),
    "RandomForest":          RandomForestClassifier(n_estimators=200, max_depth=18,
                                                    min_samples_leaf=20, n_jobs=-1, random_state=SEED),
    "HistGradientBoosting":  HistGradientBoostingClassifier(max_iter=300, learning_rate=0.1,
                                                            random_state=SEED),
}

def evaluate(name, clf):
    pipe = Pipeline([("prep", prep), ("clf", clf)])
    # poids de classe pour tous (sauf Dummy qui les ignore de toute façon)
    try:
        pipe.fit(X_tr, y_tr, clf__sample_weight=sw_tr)
    except TypeError:
        pipe.fit(X_tr, y_tr)
    if hasattr(pipe, "predict_proba"):
        proba = pipe.predict_proba(X_te)[:, 1]
    else:
        proba = pipe.decision_function(X_te)
    pred = (proba >= 0.5).astype(int) if proba.min() >= 0 and proba.max() <= 1 else pipe.predict(X_te)
    return pipe, {
        "PR-AUC":   average_precision_score(y_te, proba),
        "ROC-AUC":  roc_auc_score(y_te, proba),
        "F1 grave": f1_score(y_te, pred, zero_division=0),
        "Rappel grave": recall_score(y_te, pred, zero_division=0),
    }

results, fitted = {}, {}
for name, clf in MODELS.items():
    fitted[name], results[name] = evaluate(name, clf)
res = pd.DataFrame(results).T.round(3)
print("Comparaison (jeu de test temporel 2022-2023) :"); res

In [ ]:
base = res.loc["Dummy (plancher)", "PR-AUC"]
# Sélection : meilleur PR-AUC parmi les modèles qui apprennent (hors Dummy)
ranked = res.drop(index="Dummy (plancher)").sort_values("PR-AUC", ascending=False)
BEST = ranked.index[0]
print(f"PR-AUC plancher (Dummy) = {base:.3f}")
print(f"Modèle retenu : {BEST}  (PR-AUC={res.loc[BEST,'PR-AUC']:.3f}, "
      f"soit x{res.loc[BEST,'PR-AUC']/base:.1f} vs plancher)")

**Comment on tranche (et pourquoi pas la perf brute) :** on ne retient pas mécaniquement le PR-AUC le
plus haut. On arbitre **performance × interprétabilité × coût** :
- si l'écart `RandomForest`/`HistGB` vs `DecisionTree`/`LogReg` est faible, on **préfère le modèle le plus
  simple/transparent** (le coach valorise « un modèle adapté, pas par habitude ») ;
- comme notre livrable est l'**explication** (pas la prédiction), un modèle légèrement moins performant mais
  plus lisible peut être préférable. Cette décision est à argumenter à l'oral au vu du tableau ci-dessus.

Pour la suite, on interprète le modèle retenu via SHAP, et on **vérifie la convergence** entre modèles
(si arbre, RF et GBT pointent les mêmes leviers, le signal est robuste).

### C.3 Interprétation → leviers structurels

On décompose les prédictions avec **SHAP** (sur un échantillon — `TreeExplainer` ne passe pas à l'échelle sur
des millions de lignes). La **contribution signée** de chaque élément distingue **aggravant** (pousse vers
« grave ») de **protecteur** (pousse vers « non grave »). Un élément **protecteur absent** d'une zone à risque
devient un **candidat à installer**.

In [ ]:
import shap
# On interprète un modèle à base d'arbres (TreeExplainer) ; à défaut, RandomForest.
interpret_name = BEST if BEST in ("RandomForest", "HistGradientBoosting", "DecisionTree (whitebox)") else "RandomForest"
pipe_int = fitted[interpret_name]
prep_f   = pipe_int.named_steps["prep"]
clf_f    = pipe_int.named_steps["clf"]
feat_names = list(prep_f.get_feature_names_out())

samp = X_te.sample(min(SHAP_SAMPLE, len(X_te)), random_state=SEED)
Xs = prep_f.transform(samp)
Xs = Xs.toarray() if hasattr(Xs, "toarray") else Xs
expl = shap.TreeExplainer(clf_f)
sv = expl.shap_values(Xs)
sv = sv[1] if isinstance(sv, list) and len(sv) == 2 else (sv[..., 1] if getattr(sv, "ndim", 2) == 3 else sv)

shap_mean = pd.Series(np.asarray(sv).mean(axis=0), index=feat_names)         # contribution signée moyenne
shap_abs  = pd.Series(np.abs(np.asarray(sv)).mean(axis=0), index=feat_names) # importance
print(f"Interprétation SHAP sur '{interpret_name}' ({len(samp)} lignes)")
print("\nÉléments d'infrastructure — effet signé moyen (>0 = aggravant, <0 = protecteur) :")
infra_shap = shap_mean[[f for f in feat_names if f.startswith("infra__")]].sort_values()
infra_shap.index = [f.replace("infra__", "") for f in infra_shap.index]
print(infra_shap.round(4))

In [ ]:
# Importance globale (toutes features) — vue d'ensemble
top_imp = shap_abs.sort_values(ascending=False).head(12)
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(top_imp.index[::-1], top_imp.values[::-1])
ax.set_title(f"Importance (|SHAP| moyen) — modèle {interpret_name}")
plt.tight_layout(); plt.show()
PROTECTIVE = [el for el in INFRA if infra_shap.get(el, 0) < 0]   # protecteurs selon SHAP
AGGRAVATING = [el for el in INFRA if infra_shap.get(el, 0) > 0]
print("Aggravants (SHAP>0) :", AGGRAVATING)
print("Protecteurs (SHAP<0) :", PROTECTIVE)

**Lecture :** on attend que les **carrefours (`junction`)** ressortent côté aggravant et que les
**équipements régulateurs** (feux, passages piétons, stops, modération de trafic) ressortent côté protecteur.
C'est cohérent avec l'analyse exploratoire et c'est ce qui fonde la prescription : *là où un aléa aggravant
domine et qu'un équipement protecteur manque, on a un levier d'action.*

## D — Synthèse décisionnelle : par zone, *où* + *quoi* + *impact estimé*

On croise les deux lentilles :
- **aléa local** par zone via la **sur-représentation (lift)** — descriptif, robuste, lisible :
  `lift(el) = part pondérée de l'élément dans la zone / part au niveau du territoire` ;
- **mesure protectrice manquante** = la mesure déployable, identifiée comme protectrice (SHAP **et** effet
  conditionnel négatif), qui n'est **pas déjà présente** dans la zone ;
- **impact estimé** = `|effet conditionnel| × volume de la zone` = réduction de charge attendue, **locale par
  construction** (dépend de l'aléa, des manques propres à la zone, et de son volume).

> **Pourquoi conditionnel et pas global ?** Un effet *moyen* recommanderait le même aménagement partout
> (artefact). En conditionnant à l'aléa local, les recommandations deviennent **variées et adaptées au lieu**.

In [ ]:
# Référence territoriale : part pondérée (gravité) de chaque élément
charge_tot = zone["charge"].sum()
nat_share = {el: zone["charge_" + el].sum() / charge_tot for el in INFRA}

def local_hazard(r):
    shares = {el: (r["charge_" + el] / r["charge"]) / nat_share[el] if nat_share[el] > 0 else 0.0
              for el in INFRA}
    top = max(shares, key=shares.get)
    return (top, shares[top]) if shares[top] >= LIFT_MIN else ("diffus", shares[top])

priority[["dominant", "lift"]] = priority.apply(
    lambda r: pd.Series(local_hazard(r)), axis=1)
print("Aléa local dominant (lift) — distribution :")
print(priority["dominant"].value_counts())

In [ ]:
# Effet CONDITIONNEL d'une mesure P là où l'aléa H est présent (sur le territoire) :
#   delta(P|H) = sévérité(H=1, P=1) - sévérité(H=1, P=0)   (<0 => P aide en présence de H)
DEPLOYABLE = ["traffic_signal", "crossing", "stop", "traffic_calming"]   # leviers réalistes du DOT
LABELS = {"traffic_signal": "installer des feux",
          "crossing": "sécuriser passage piéton",
          "stop": "stop / cédez-le-passage",
          "traffic_calming": "modération de trafic (ralentisseurs)"}

def cond_delta(H, P):
    sub = terr[terr[H] == 1]
    a, b = sub[sub[P] == 0]["severity"], sub[sub[P] == 1]["severity"]
    if len(a) < MIN_N or len(b) < MIN_N:
        return np.nan
    return b.mean() - a.mean()

cond = {H: {P: cond_delta(H, P) for P in DEPLOYABLE if P != H} for H in INFRA}

def recommend(r):
    present = {el: r["n_" + el] / r["n_accidents"] for el in INFRA}
    H = r["dominant"]
    if H == "diffus":
        return pd.Series({"amenagement": "investiguer facteur humain (vitesse / contrôle)",
                          "impact_charge": 0.0})
    ranked = sorted([(P, cond[H][P]) for P in DEPLOYABLE
                     if P != H and pd.notna(cond[H].get(P)) and cond[H][P] < 0], key=lambda x: x[1])
    missing = [(P, v) for P, v in ranked if present[P] < PRESENCE_MIN]
    if not missing:
        return pd.Series({"amenagement": "déjà équipé — revoir géométrie / abords", "impact_charge": 0.0})
    P, v = missing[0]
    label = "sécuriser le passage à niveau (barrières/feux)" if H == "railway" else LABELS[P]
    return pd.Series({"amenagement": label, "impact_charge": abs(v) * r["n_accidents"]})

priority[["amenagement", "impact_charge"]] = priority.apply(recommend, axis=1)
print("Aménagements recommandés (top zones) :")
print(priority["amenagement"].value_counts())

In [ ]:
# Livrable : tableau trié par impact estimé (= ordre d'intervention proposé) + export CSV
def present_list(r):
    p = [el for el in INFRA if r["n_" + el] / r["n_accidents"] >= PRESENCE_MIN]
    return ", ".join(p) if p else "—"

reco = priority.copy()
reco["equip_present"] = reco.apply(present_list, axis=1)
reco = reco.sort_values("impact_charge", ascending=False)
out_cols = ["rank", "h3_cell", "n_accidents", "charge", "avg_severity",
            "street_km", "charge_per_km", "dominant", "lift",
            "equip_present", "amenagement", "impact_charge"]
out = reco[out_cols].round({"avg_severity": 3, "lift": 2, "street_km": 2,
                            "charge_per_km": 2, "impact_charge": 0})
out.to_csv("zones_prioritaires.csv", index=False)
print("Export : zones_prioritaires.csv —", len(out), "zones")
out.head(15)

In [ ]:
# Carte : pastilles dimensionnées par le risque, couleur = aménagement recommandé.
import folium
actions = sorted(reco["amenagement"].unique())
palette = ["#e74c3c", "#3498db", "#2ecc71", "#9b59b6", "#f39c12", "#1abc9c", "#e67e22", "#34495e"]
color_of = {a: palette[i % len(palette)] for i, a in enumerate(actions)}
size_col = "charge_per_km" if reco["charge_per_km"].notna().any() else "charge"
size_max = reco[size_col].max()

m = folium.Map(tiles="CartoDB positron")
centers = []
for _, r in reco.iterrows():
    try:
        c = list(h3_center(r["h3_cell"])); centers.append(c)
        col = color_of[r["amenagement"]]
        val = r[size_col] if pd.notna(r[size_col]) else r["charge"]
        radius = 5 + 18 * (val / size_max if size_max else 0)
        tip = (f"<b>#{int(r['rank'])}</b> — {int(r['n_accidents']):,} accidents<br>"
               f"Aléa local : <b>{r['dominant']}</b><br>Présent : {r['equip_present']}<br>"
               f"➜ <b>{r['amenagement']}</b><br>Impact estimé : {r['impact_charge']:.0f}")
        b = h3_boundary(r["h3_cell"])
        folium.Polygon([(la, ln) for la, ln in b] + [(b[0][0], b[0][1])],
                       color=col, weight=1, fill=True, fill_color=col, fill_opacity=0.25).add_to(m)
        folium.CircleMarker(c, radius=radius, color="#222", weight=1, fill=True,
                            fill_color=col, fill_opacity=0.9,
                            tooltip=folium.Tooltip(tip)).add_to(m)
    except Exception:
        pass
if centers:
    m.fit_bounds(centers)
items = "".join(f"<div><span style='background:{color_of[a]};width:12px;height:12px;"
                f"display:inline-block;margin-right:6px;'></span>{a}</div>" for a in actions)
legend = (f"<div style='position:fixed;bottom:30px;left:30px;z-index:9999;background:white;"
          f"padding:10px;border:1px solid #999;font-size:11px;max-width:320px;'>"
          f"<b>Aménagement recommandé</b><br><i>taille = risque ({size_col})</i>{items}</div>")
m.get_root().html.add_child(folium.Element(legend))
m.save("carte_zones_prioritaires.html")
print("Carte : carte_zones_prioritaires.html")
m

## E — Analyse critique & recul

### Ce qui fonctionne
- **Priorisation objective** : un classement des zones reproductible, indépendant des accidents médiatisés,
  que le DOT peut défendre avec des données.
- **Convergence des signaux** : quand l'aléa local (lift, descriptif) et les contributions SHAP (modèle)
  pointent le même élément, le diagnostic est robuste (double preuve).
- **Exposition OSM** : distingue « dangereux » de « fréquenté » — corrige un biais classique du comptage brut.
- **Comparaison de modèles** : le choix est argumenté sur une métrique honnête, pas pris « par habitude ».

### Ce qui ne fonctionne pas / reste incertain
- **Zones « diffuses »** : certaines zones n'ont pas d'aléa infra dominant → le modèle ne désigne pas de
  levier structurel clair (probable facteur humain : vitesse, contrôle). On l'**affiche** au lieu de le masquer.
- **Pouvoir prédictif modéré** : prédire un accident *grave* à partir du seul contexte infra+météo+heure reste
  difficile (PR-AUC à comparer au plancher). C'est attendu : la gravité dépend aussi de facteurs absents des
  données (vitesse réelle, comportement). Le modèle est un **révélateur de leviers**, pas un oracle.
- **Désaccords A/B** : si lift et SHAP divergent sur une zone, c'est un signal que l'exposition fausse l'un des
  deux ; à documenter, pas à trancher arbitrairement.

### Limites assumées (« deuils »)
- **Pas de causalité** : aucune donnée avant/après installation → associations, pas preuves d'efficacité.
- **Pas de profil conducteur** (âge, alcool, vitesse), pas de **coût réel** (`severity` = seul proxy).
- **Biais de reporting** (MapQuest/Bing) : couverture inégale entre zones.
- **Univers = lieux déjà accidentogènes** : on explique « parmi les lieux à accidents, quoi aggrave »
  (l'exposition OSM atténue mais ne supprime pas ce biais).

### Périmètre d'usage du modèle
- ✅ Prioriser des **études d'aménagement** sur un territoire à fort volume de données.
- ✅ Argumenter un **classement** de zones et un **type d'intervention** plausible.
- ❌ Garantir qu'un aménagement *réduira* les accidents (nécessiterait une évaluation ex-post).
- ❌ Conclure sur des zones à très faible volume (puissance statistique insuffisante).

### Décision concrète sur le terrain
Le livrable `zones_prioritaires.csv` + la carte donnent au DOT, **lundi matin**, une liste ordonnée :
*« commencer par la zone #X (carrefour sans feux, N accidents/an) → lancer une étude d'installation de feux ;
impact attendu ≈ … »*. C'est une **aide à la priorisation**, validée ensuite par une visite terrain.

### Perspectives (si plus de temps)
- **Causalité** : panel avant/après, ou méthodes quasi-expérimentales.
- **Exposition complète** : trafic AADT + OSM sur *toutes* les cellules → vrais taux d'accident.
- **Risque conjoncturel** : modèle dédié « QUAND » pour piloter des mesures dynamiques.
- **Industrialisation** : packaging `src/` + tests + Docker (Java/Spark encapsulés).